# COSMIC v3.3 Signature Analysis

Analysis of SBS96 signature assignments in 52 matched mammalian species across three mitochondrial genes (CO1, CO3, Cytb).

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Setup
HERE = Path(".").resolve()
RESULTS_DIR = HERE / "results"
FIGURES_DIR = HERE / "figures"
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

# Configuration
GENES = ("CO1", "CO3", "Cytb")
GENE_LABELS = {"CO1": "COX1", "CO3": "COX3", "Cytb": "CytB"}
BRANCHES = ("low_Ts", "high_Ts", "high_minus_low_Ts")
BRANCH_LABELS = {
    "low_Ts": "Low transitions",
    "high_Ts": "High transitions",
    "high_minus_low_Ts": "High - low transitions",
}
SPECTRUM_TYPES = ("Ts_only", "Ts_and_Tv")

SIGNATURE_COLORS = {
    "SBS12": "#0072B2",
    "SBS30": "#E69F00",
    "SBS23": "#009E73",
    "SBS26": "#CC79A7",
    "SBS19": "#56B4E9",
    "SBS42": "#D55E00",
    "SBS5": "#6F4C9B",
    "SBS44": "#F0C808",
    "SBS1": "#8C564B",
    "SBS2": "#17BECF",
    "Other": "#B5BDC7",
}

print(f"Genes: {GENES}")
print(f"Branches: {BRANCHES}")
print(f"Spectrum types: {SPECTRUM_TYPES}")
print(f"Total samples: {len(GENES) * len(BRANCHES) * len(SPECTRUM_TYPES)}")

## 1. Load Results from Both Tools

In [ ]:
# Load SigProfilerAssignment results
sigprofiler = pd.read_csv(RESULTS_DIR / "sigprofiler_activities.csv")
print(f"SigProfilerAssignment activities: {len(sigprofiler)} rows")
print(f"Genes: {sorted(sigprofiler['Gene'].unique())}")
print(f"Branches: {sorted(sigprofiler['Branch'].unique())}")
print(f"Spectrum types: {sorted(sigprofiler['SpectrumType'].unique())}")
print(f"\nTop signatures by total activity:")
print(sigprofiler.groupby("Signature")["Activity"].sum().sort_values(ascending=False).head(10))

In [ ]:
# Load mSigAct results
msigact = pd.read_csv(RESULTS_DIR / "msigact_activities.csv")
print(f"mSigAct activities: {len(msigact)} rows")
print(f"Available modes: {sorted(msigact['Mode'].unique())}")

# Select primary mode
PREFERRED_MSIGACT_MODES = ("uniform_top6", "sigprofiler_priors", "uniform")
available_modes = set(msigact["Mode"])
selected_mode = next(
    (mode for mode in PREFERRED_MSIGACT_MODES if mode in available_modes),
    None,
)
if selected_mode is None:
    raise ValueError(f"No supported mSigAct mode found: {sorted(available_modes)}")
print(f"\nSelected mSigAct mode: {selected_mode}")

## 2. Normalize and Combine Results

In [ ]:
# Prepare SigProfiler data
sigprofiler_clean = sigprofiler.loc[:, ["Branch", "Gene", "SpectrumType", "Signature", "Activity"]].copy()
sigprofiler_clean["Tool"] = "SigProfilerAssignment"

# Prepare mSigAct data
msigact_clean = msigact.loc[
    msigact["Mode"].eq(selected_mode),
    ["Branch", "Gene", "SpectrumType", "Signature", "Activity"]
].copy()
msigact_clean["Tool"] = f"mSigAct ({selected_mode})"

# Combine and normalize
activities = pd.concat([sigprofiler_clean, msigact_clean], ignore_index=True)
activities["Activity"] = pd.to_numeric(activities["Activity"], errors="raise")

# Normalize within samples
totals = activities.groupby(
    ["Tool", "Branch", "Gene", "SpectrumType"], observed=True
)["Activity"].transform("sum")
activities["ActivityFraction"] = activities["Activity"] / totals

print(f"Combined activities: {len(activities)} rows")
print(f"Tools: {sorted(activities['Tool'].unique())}")

## 3. Calculate Cross-Tool Concordance

In [ ]:
def calculate_tool_concordance(activities, msigact_mode):
    tools = list(pd.unique(activities["Tool"]))
    index_columns = ["Branch", "Gene", "SpectrumType"]
    rows = []
    
    for key, sample in activities.groupby(index_columns, sort=False, observed=True):
        wide = sample.pivot_table(
            index="Signature",
            columns="Tool",
            values="ActivityFraction",
            aggfunc="sum",
            fill_value=0.0,
        ).reindex(columns=tools, fill_value=0.0)
        
        first = wide.iloc[:, 0].to_numpy(dtype=float)
        second = wide.iloc[:, 1].to_numpy(dtype=float)
        norm_product = float(np.linalg.norm(first) * np.linalg.norm(second))
        cosine = float(np.dot(first, second) / norm_product) if norm_product else np.nan
        total_variation = float(0.5 * np.abs(first - second).sum())
        
        rows.append({
            **dict(zip(index_columns, key)),
            "mSigActMode": msigact_mode,
            "CosineSimilarityBetweenTools": cosine,
            "TotalVariationDistance": total_variation,
            "DominantSigProfilerSignature": str(wide.iloc[:, 0].idxmax()),
            "DominantMSigActSignature": str(wide.iloc[:, 1].idxmax()),
            "SameDominantSignature": bool(wide.iloc[:, 0].idxmax() == wide.iloc[:, 1].idxmax()),
        })
    
    result = pd.DataFrame(rows).sort_values(index_columns).reset_index(drop=True)
    return result

concordance = calculate_tool_concordance(activities, selected_mode)
print(f"Cross-tool concordance statistics:")
print(concordance[[
    "CosineSimilarityBetweenTools",
    "TotalVariationDistance",
    "SameDominantSignature"
]].agg(["min", "median", "max"]).to_string())

## 4. Prepare Data for Visualization

In [ ]:
def condense_signatures(activities):
    mean_fraction = activities.groupby("Signature")["ActivityFraction"].mean()
    keep = set(mean_fraction.index[mean_fraction.ge(0.01)])
    
    # Always retain dominant signature of every sample
    dominant = (
        activities.sort_values("ActivityFraction", ascending=False)
        .groupby(["Tool", "Branch", "Gene", "SpectrumType"], observed=True)
        .head(1)["Signature"]
    )
    keep.update(dominant)
    
    condensed = activities.copy()
    condensed["DisplaySignature"] = np.where(
        condensed["Signature"].isin(keep), condensed["Signature"], "Other"
    )
    condensed = (
        condensed.groupby(
            ["Tool", "Branch", "Gene", "SpectrumType", "DisplaySignature"],
            as_index=False,
            observed=True,
        )["ActivityFraction"]
        .sum()
    )
    
    signature_order = sorted(
        keep,
        key=lambda signature: (
            -mean_fraction.get(signature, 0.0),
            int("".join(filter(str.isdigit, signature)) or 10_000),
            signature,
        ),
    )
    if "Other" in set(condensed["DisplaySignature"]):
        signature_order.append("Other")
    
    return condensed, signature_order

condensed, signature_order = condense_signatures(activities)
print(f"Display signatures (>1% or dominant): {signature_order}")

## 5. Signature Activity Comparison by Gene

In [ ]:
def plot_activity_comparison(activities, signature_order, selected_mode):
    def signature_color(signature, fallback_index):
        if signature in SIGNATURE_COLORS:
            return SIGNATURE_COLORS[signature]
        palette = mpl.colormaps["tab20"]
        return mpl.colors.to_hex(palette((fallback_index % 20) / 19))
    
    tools = list(pd.unique(activities["Tool"]))
    sample_order = [(gene, branch) for gene in GENES for branch in ("high_Ts", "low_Ts", "high_minus_low_Ts")]
    x = np.arange(len(sample_order))
    branch_short = {"high_Ts": "High", "low_Ts": "Low", "high_minus_low_Ts": "High-low"}
    
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": 9.2,
        "axes.labelcolor": "#20262E",
        "axes.titlecolor": "#20262E",
        "xtick.color": "#20262E",
        "ytick.color": "#20262E",
        "pdf.fonttype": 42,
        "savefig.facecolor": "white",
    })
    
    fig, axes = plt.subplots(2, 2, figsize=(13.2, 8.4), sharex=True, sharey=True)
    colors = {
        signature: signature_color(signature, index)
        for index, signature in enumerate(signature_order)
    }
    
    for row_index, tool in enumerate(tools):
        for column_index, spectrum_type in enumerate(SPECTRUM_TYPES):
            axis = axes[row_index, column_index]
            subset = activities.loc[
                activities["Tool"].eq(tool)
                & activities["SpectrumType"].eq(spectrum_type)
            ]
            bottom = np.zeros(len(sample_order), dtype=float)
            
            for signature in signature_order:
                values = []
                for gene, branch in sample_order:
                    selected = subset.loc[
                        subset["Gene"].eq(gene)
                        & subset["Branch"].eq(branch)
                        & subset["DisplaySignature"].eq(signature),
                        "ActivityFraction",
                    ]
                    values.append(float(selected.sum()))
                values_array = np.asarray(values)
                axis.bar(
                    x, values_array, bottom=bottom, width=0.72,
                    color=colors[signature], edgecolor="white", linewidth=0.35,
                    label=signature,
                )
                bottom += values_array
            
            for separator in (2.5, 5.5):
                axis.axvline(separator, color="#A7AFB8", linewidth=0.8)
            
            axis.set_ylim(0, 1)
            axis.set_yticks(np.linspace(0, 1, 6))
            axis.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
            axis.grid(axis="y", color="#D9DEE4", linewidth=0.55, alpha=0.75)
            axis.set_axisbelow(True)
            axis.spines[["top", "right", "left"]].set_visible(False)
            axis.tick_params(axis="y", length=0)
            axis.set_title(
                "Transitions only" if spectrum_type == "Ts_only"
                else "Transitions + transversions",
                loc="left", fontsize=10.4, fontweight="bold",
            )
            if column_index == 0:
                axis.set_ylabel(f"{tool}\nAssigned activity")
            axis.text(
                -0.095, 1.08, chr(65 + row_index * 2 + column_index),
                transform=axis.transAxes, fontsize=13, fontweight="bold", va="top",
            )
    
    labels = [
        f"{GENE_LABELS.get(gene, gene)}\n{branch_short[branch]}"
        for gene, branch in sample_order
    ]
    for axis in axes[-1, :]:
        axis.set_xticks(x, labels, rotation=45, ha="right")
        axis.tick_params(axis="x", length=0, pad=5)
    
    handles = [
        Patch(facecolor=colors[signature], label=signature)
        for signature in signature_order
    ]
    fig.legend(
        handles=handles, loc="center left", bbox_to_anchor=(0.875, 0.50),
        frameon=False, title="COSMIC SBS96", ncol=1,
    )
    
    fig.suptitle(
        "COSMIC v3.3 signature activities in matched mammalian gene spectra",
        x=0.075, y=0.995, ha="left", fontsize=15, fontweight="bold",
    )
    fig.text(
        0.075, 0.957,
        "52 matched species · human GRCh37 context renormalization · "
        f"mSigAct mode: {selected_mode}",
        color="#5F6872", fontsize=9.2,
    )
    fig.subplots_adjust(left=0.09, right=0.86, bottom=0.13, top=0.89, hspace=0.30, wspace=0.14)
    
    output = FIGURES_DIR / "signature_activities_by_gene"
    fig.savefig(output.with_suffix(".png"), dpi=400, bbox_inches="tight")
    fig.savefig(output.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {output}.png")

plot_activity_comparison(condensed, signature_order, selected_mode)

## 6. Gene Comparison: Signature Activities by Branch

In [ ]:
# Compare signature activities across genes within each branch
for branch in BRANCHES:
    branch_data = condensed.loc[
        (condensed["Branch"] == branch) & 
        (condensed["SpectrumType"] == "Ts_and_Tv")
    ]
    
    sigprof = branch_data.loc[branch_data["Tool"] == "SigProfilerAssignment"]
    print(f"\n{BRANCH_LABELS[branch]} (SigProfilerAssignment):")
    for gene in GENES:
        gene_sigs = sigprof.loc[sigprof["Gene"] == gene].sort_values(
            "ActivityFraction", ascending=False
        )
        top_sigs = gene_sigs[gene_sigs["ActivityFraction"] >= 0.01]
        print(f"  {GENE_LABELS[gene]}: {len(top_sigs)} signatures, "
              f"top: {top_sigs.iloc[0]['DisplaySignature'] if len(top_sigs) > 0 else 'N/A'} "
              f"({top_sigs.iloc[0]['ActivityFraction']:.2%})")

## 7. Export Results

In [ ]:
# Save normalized activities
activities.to_csv(RESULTS_DIR / "tool_activity_fractions.csv", index=False)
concordance.to_csv(RESULTS_DIR / "tool_concordance.csv", index=False)

print("Exported results:")
print(f"  - {RESULTS_DIR / 'tool_activity_fractions.csv'}")
print(f"  - {RESULTS_DIR / 'tool_concordance.csv'}")
print(f"\nFigures:")
print(f"  - {FIGURES_DIR / 'signature_activities_by_gene.png'}")
print(f"  - {FIGURES_DIR / 'signature_activities_by_gene.pdf'}")

## 8. Summary Statistics

In [ ]:
print(f"\n{'='*60}")
print(f"ANALYSIS SUMMARY")
print(f"{'='*60}")
print(f"\nGenes analyzed: {', '.join(GENES)}")
print(f"Branches: {', '.join(BRANCHES)}")
print(f"Total spectra: {len(GENES) * len(BRANCHES) * len(SPECTRUM_TYPES)}")
print(f"\nSignatures identified (>1% or dominant): {len(signature_order)}")
print(f"\nCross-tool agreement (Cosine similarity):")
print(f"  Min: {concordance['CosineSimilarityBetweenTools'].min():.3f}")
print(f"  Median: {concordance['CosineSimilarityBetweenTools'].median():.3f}")
print(f"  Max: {concordance['CosineSimilarityBetweenTools'].max():.3f}")
print(f"\nSame dominant signature: {concordance['SameDominantSignature'].sum()}/{len(concordance)} samples")
print(f"Percentage: {100 * concordance['SameDominantSignature'].mean():.1f}%")